In [44]:
!pip install --upgrade --quiet pinecone pinecone-text pinecone-notebooks


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [45]:
api_key = "pcsk_LPywF_3NTFJoGDbC6cuqEsKp8wMBLdp4Fmr1VXALBtdN1wLcWBH7QF7CTe4dJkxxRzd55"

In [46]:
from langchain_community.retrievers import PineconeHybridSearchRetriever

In [47]:
import os
from pinecone import Pinecone, ServerlessSpec
index_name=  "hybrid-search-langchain-pinecone"
pc = Pinecone(api_key=api_key)


# below can be done using website UI
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=1536, # use the dimensin of my embedding model
        metric='dotproduct', #sparse values supported only for dotproduct
        spec=ServerlessSpec(cloud='aws', region="us-east-1")
    )



In [48]:
index = pc.Index(index_name)
index

In [49]:
# vector embedding + sparse matric = hybrid
from dotenv import load_dotenv
load_dotenv
from openai import OpenAI

client = OpenAI()

OPENAI_API_KEY=os.getenv("OPENAI_API_KEY")



from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model = "text-embedding-3-small")
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x168437360>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x32fac4640>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [50]:
from pinecone_text.sparse import BM25Encoder # use tfidf bydefault
bm25_encoder = BM25Encoder().default()
bm25_encoder

In [88]:
sentences = [
    "Hybrid retrieval combines dense and sparse search",
    "It uses embeddings to capture semantic meaning",
    "It uses keyword methods like BM25 for exact matching",
    "Dense search understands paraphrases and context",
    "Sparse search matches exact words and numbers",
    "Hybrid improves both precision and recall",
    "Scores from both methods are combined together",
    "It reduces failure on rare terms and codes",
    "It works well for production RAG systems",
    "It is more robust than using only one method"
]

In [89]:
bm25_encoder.fit(sentences)

100%|██████████| 10/10 [00:00<00:00, 9283.54it/s]


In [90]:
bm25_encoder.dump("bm25_values.json")
bm25_encoder = BM25Encoder().load("bm25_values.json")

In [98]:
retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings, # dense vector
    sparse_encoder=bm25_encoder, # sparce matrix
    index= index,
    alpha=0.5
)

In [99]:
retriever

PineconeHybridSearchRetriever(embeddings=OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x168437360>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x32fac4640>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x16d7ad090>, index=<pinecone.db_data.index.Index object at 0x310

In [100]:
retriever.add_texts(sentences)

100%|██████████| 1/1 [00:01<00:00,  1.36s/it]


In [ ]:
# Retrieve
docs = retriever.invoke("hybrid search")

# Extract text
context = "\n".join(doc.page_content for doc in docs)

In [108]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)


query = "What is a transformers?"

# Retrieve
docs = retriever.invoke(query)

# Extract text
context = "\n".join(doc.page_content for doc in docs)

# Generate answer
prompt = f"""
You are a QA system.

STRICT RULES:
1. Use ONLY the provided context.
2. If the answer is not explicitly in the context, say:
   "I don't know based on the provided context."
3. Do NOT use outside knowledge.

Context:
{context}

Question:
{query}

Answer:
"""



In [109]:
from IPython.display import display, Markdown

response = llm.invoke(prompt)

display(Markdown(response.content))

I don't know based on the provided context.